# Endgame Quickstart

This notebook demonstrates a complete ML workflow using Endgame:
1. Load a dataset from OpenML
2. Train a LightGBM model with competition-winning defaults
3. Evaluate performance with cross-validation metrics
4. Generate an interactive HTML evaluation report
5. Export a standalone Python script

In [ ]:
import endgame as eg
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

## 1. Load Data

We'll use the German Credit dataset (OpenML #31) — a classic binary classification task
with mixed numeric and categorical features.

In [ ]:
credit = fetch_openml(data_id=31, as_frame=True, parser="auto")
df = credit.frame
print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['class'].value_counts()}")
df.head()

In [ ]:
X = df.drop(columns=["class"])
y = df["class"]

# Encode categorical columns as integers for LightGBM compatibility
from sklearn.preprocessing import OrdinalEncoder
cat_cols = X.select_dtypes(include="category").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 2. Train a Model

Endgame's `LGBMWrapper` wraps LightGBM with competition-tuned presets.
The `'endgame'` preset uses a low learning rate (0.01), 10,000 max trees,
and early stopping — the same configuration that wins Kaggle competitions.

In [ ]:
model = eg.models.LGBMWrapper(preset="endgame")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
)
print(f"Test accuracy: {model.score(X_test, y_test):.4f}")

## 3. Evaluate

Check predictions and feature importances.

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Feature importances
importances = model.feature_importances_  # dict: {feature_name: importance}

# Sort by importance (descending)
sorted_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)

print("Top 10 features:")
for name, imp in sorted_features[:10]:
    print(f"  {name:>30s}: {imp:.0f}")

## 4. Generate an Evaluation Report

Endgame generates self-contained HTML reports with metrics, confusion matrix,
ROC curve, precision-recall curve, calibration plot, and feature importances.

In [ ]:
from endgame.visualization import ClassificationReport

report = ClassificationReport(
    model, X_test, y_test.values,
    model_name="LightGBM (endgame preset)",
    dataset_name="German Credit",
)
report.save("quickstart_report.html")
print("Report saved to quickstart_report.html")

## 5. Save the Model

Endgame's persistence module saves the model with metadata for reproducibility.

In [ ]:
eg.save(model, "german_credit_lgbm")
print("Model saved.")

# Later, reload with:
# model = eg.load("german_credit_lgbm")

## Summary

In this notebook we:
- Loaded the German Credit dataset from OpenML
- Trained a LightGBM model with Endgame's competition-winning defaults
- Evaluated with sklearn metrics
- Generated a self-contained HTML evaluation report
- Saved the model for later use

Next steps:
- Try `02_interpretable_models.ipynb` for glass-box models
- Try `03_automl.ipynb` for automated model selection and ensembling